# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 06 · What actually helped in Round 1?

**NFL trajectory feature research · arrival attribution · CPU only**

Round 1 found a **3.59%** pooled gain from arrival features and a **13.11% deterioration** from the earlier-origin mixture. We stop that mixture and decompose the useful arrival representation.

This notebook reuses six existing control/arrival fits and makes at most **18 small new fits**. It never fits the established neural model or evaluates Kaggle. Use the existing **NFL Trajectory (Python 3.11)** kernel. Run each cell in order; stop on any failure. Do not run terminal stages concurrently.

The two opening charts are measured **Round 1** aggregate evidence. Later charts require your new stage outputs; no placeholder scores are substituted.

In [ ]:
from pathlib import Path
import json
import sys
import subprocess
import pandas as pd
import plotly.io as pio

KIT = Path('/home/sagemaker-user/nfl_feature_round2')
OUT = Path('/home/sagemaker-user/nfl-feature-round2-results')
if not KIT.is_dir():
    raise FileNotFoundError('Upload and extract nfl_feature_round2.zip into /home/sagemaker-user first.')
sys.path.insert(0, str(KIT))
import visuals
pio.renderers.default = 'plotly_mimetype'
HTML = OUT / 'visualizations'

def run_stage(command, seconds):
    subprocess.run([sys.executable, str(KIT / 'run_round.py'), command,
                    '--out', str(OUT), '--seconds', str(seconds)], cwd=KIT, check=True)

def read_result(relative):
    path = OUT / relative
    if not path.is_file():
        raise FileNotFoundError(f'Missing stage result: {path}. Run the preceding stage; do not invent results.')
    return json.loads(path.read_text())

print('Kernel:', sys.executable)
print('Outputs:', OUT)
print('No new cloud jobs, package installation, Git writes or Kaggle submissions.')


## 1 · Recovered evidence, not another training run

In [ ]:
prior = json.loads((KIT / 'evidence/round1_screen_summary.json').read_text())
visuals.show_save(visuals.prior_scores(prior), HTML, '01_round1_scores')

In [ ]:
visuals.show_save(visuals.prior_fold_gains(prior), HTML, '02_round1_fold_stability')

## 2 · Verify exact parent-model replay
120-second hard cap. This checks old source/data lineage and numerical predictions, with **zero parent refits**. A mismatch stops; do not recreate the environment to disguise it.

In [ ]:
run_stage('preflight', 120)
preflight = read_result('preflight.json')
assert preflight['status'] == 'parent_replay_verified'
pd.DataFrame([preflight])

## 3 · Attribute the arrival gain

For each subgroup, test **control + subgroup** and **full arrival − subgroup**: required-velocity response, quadratic arrival response, and receiver-relative velocity. Same rows, same targets, same ridge penalty. The subgroups can be correlated; addition and removal answer different questions.

180-second cap. **18 new fits maximum**, three reused chronological folds. No parameter search. Completed model checkpoints resume without retraining.

In [ ]:
run_stage('attribute', 180)
result = read_result('attribution/summary.json')
assert result['status'] == 'attribution_complete'
display(pd.DataFrame(result['decisions']))

In [ ]:
visuals.show_save(visuals.contrast_intervals(result), HTML, '03_arrival_contribution')

In [ ]:
visuals.show_save(visuals.fold_scores(result), HTML, '04_arrival_fold_scores')

## 4 · Interpretation and checkpoint

For additions, negative differences favor the candidate. For removals, a positive difference shows a cost of removing that family. The interval gate adjusts for all **nine planned Round 2 comparisons**, not just the most favorable arm. The folds were already used in Round 1, so these remain exploratory findings, not independent confirmation.

No feature is promoted into your established model by this notebook. Continue with `07_turning_braking_features.ipynb` only after this notebook completes. An interval crossing zero is a scientific result—not a reason to rerun with different settings.

In [ ]:
run_stage('report', 60)
print('Partial return report:', OUT / 'nfl_feature_round2_report.zip')
print('Save this notebook. Next: 07_turning_braking_features.ipynb.')